In [1]:
# CELL 1: LIBRARIES & IMPORTS
import os, re, math, glob
import numpy as np
import nibabel as nib
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule


print("✅ Environment ready.")

✅ Environment ready.


In [2]:
# ==============================================================================
#  UMER-ENGINE: SYNTHETIC METADATA GENERATOR (FINAL FIXED VERSION)
#  Generates: "umer_synthetic_data.json" for Unity Import
# ==============================================================================

import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import numpy as np
import json
import random
import math

# --- 1. CUDA PHYSICS KERNEL (WITH EXTERN C WRAPPER) ---
CUDA_SOURCE = """
#include <stdint.h> 

#define HASH_SIZE 65536
#define EMPTY_KEY 0xFFFFFFFFFFFFFFFF

struct Voxel {
    float vx, vy, vz;
    float sub_x, sub_y, sub_z;
};

// C++ Helper functions (Can remain mangled, GPU only)
__device__ unsigned long long pack(int x, int y, int z) { 
    return ((unsigned long long)(x+10000)&0xFFFFF) | 
           (((unsigned long long)(y+10000)&0xFFFFF)<<20) | 
           (((unsigned long long)(z+10000)&0xFFFFF)<<40); 
}

__device__ void unpack(unsigned long long k, int* x, int* y, int* z) { 
    *x=(int)(k&0xFFFFF)-10000; 
    *y=(int)((k>>20)&0xFFFFF)-10000; 
    *z=(int)((k>>40)&0xFFFFF)-10000; 
}

// THE FIX: extern "C" prevents name mangling so Python can find the function
extern "C" {
    __global__ void update_physics(
        unsigned long long* src_keys, int* src_vals, Voxel* src_heap,
        unsigned long long* dst_keys, int* dst_vals, Voxel* dst_heap,
        int max_voxels, float dt
    ) {
        int idx = blockIdx.x * blockDim.x + threadIdx.x;
        if (idx >= max_voxels) return;

        unsigned long long key = src_keys[idx];
        if (key == EMPTY_KEY) return;
        
        int heap_idx = src_vals[idx];
        Voxel me = src_heap[heap_idx];
        
        // Simple Gravity
        me.vz -= 9.8f * dt; 
        
        // Floor Collision (Bounce)
        int x, y, z;
        unpack(key, &x, &y, &z);
        
        if (z <= 0 && me.vz < 0) {
            me.vz *= -0.6f; // Bouncy
            me.vx *= 0.9f;  // Friction
            me.vy *= 0.9f;
        }

        // Position Update
        me.sub_x += me.vx * dt;
        me.sub_y += me.vy * dt;
        me.sub_z += me.vz * dt;
        
        int dx = (int)me.sub_x;
        int dy = (int)me.sub_y;
        int dz = (int)me.sub_z;
        me.sub_x -= dx; me.sub_y -= dy; me.sub_z -= dz;
        
        int tx = x + dx; int ty = y + dy; int tz = z + dz;
        if (tz < 0) tz = 0; // Hard floor clamp

        // Output
        unsigned long long new_key = pack(tx, ty, tz);
        
        // Hash Insert (Linear Probe)
        // Explicit cast to unsigned int to satisfy nvcc
        unsigned int h = (unsigned int)(new_key % HASH_SIZE);
        
        for(int i=0; i<32; i++) {
            int slot = (h + i) % HASH_SIZE;
            unsigned long long old = atomicCAS(&dst_keys[slot], EMPTY_KEY, new_key);
            if (old == EMPTY_KEY || old == new_key) {
                dst_vals[slot] = heap_idx;
                dst_heap[heap_idx] = me;
                break;
            }
        }
    }
}
"""

class UmerEngine:
    def __init__(self):
        # no_extern_c=True allows us to write our own extern "C" block
        # This is required when we include headers like <stdint.h>
        self.mod = SourceModule(CUDA_SOURCE, no_extern_c=True)
        self.kernel = self.mod.get_function("update_physics")
        self.hash_size = 65536
        self.cap = 10000
        
        # GPU Alloc
        self.keys_A = cuda.mem_alloc(self.hash_size * 8)
        self.vals_A = cuda.mem_alloc(self.hash_size * 4)
        self.keys_B = cuda.mem_alloc(self.hash_size * 8)
        self.vals_B = cuda.mem_alloc(self.hash_size * 4)
        self.heap_A = cuda.mem_alloc(self.cap * 24) 
        self.heap_B = cuda.mem_alloc(self.cap * 24)
        
        self.host_keys = np.full(self.hash_size, 0xFFFFFFFFFFFFFFFF, dtype=np.uint64)
        self.host_vals = np.full(self.hash_size, -1, dtype=np.int32)
        
        # Structured Array
        self.host_heap = np.zeros(self.cap, dtype=np.dtype([
            ('vx','f4'),('vy','f4'),('vz','f4'),
            ('sx','f4'),('sy','f4'),('sz','f4')
        ]))
        
        self.active_count = 0
        self.frame = 0

    def reset(self):
        cuda.memset_d32(self.keys_A, 0xFFFFFFFF, self.hash_size * 2)
        self.active_count = 0
        self.frame = 0

    def add_object(self, cx, cy, cz, size=10):
        for x in range(cx, cx+size):
            for y in range(cy, cy+size):
                for z in range(cz, cz+size):
                    if self.active_count >= self.cap: return
                    idx = self.active_count
                    
                    self.host_heap[idx]['vx'] = random.uniform(-5, 5)
                    self.host_heap[idx]['vy'] = random.uniform(-5, 5)
                    self.host_heap[idx]['vz'] = 0
                    
                    k = ((x+10000)&0xFFFFF) | (((y+10000)&0xFFFFF)<<20) | (((z+10000)&0xFFFFF)<<40)
                    h = k % self.hash_size
                    for i in range(32):
                        slot = int((h+i)%self.hash_size)
                        if self.host_keys[slot] == 0xFFFFFFFFFFFFFFFF:
                            self.host_keys[slot] = k
                            self.host_vals[slot] = idx
                            break
                    self.active_count += 1
        
        cuda.memcpy_htod(self.keys_A, self.host_keys)
        cuda.memcpy_htod(self.vals_A, self.host_vals)
        cuda.memcpy_htod(self.heap_A, self.host_heap)

    def step(self):
        in_k = self.keys_A if self.frame%2==0 else self.keys_B
        in_v = self.vals_A if self.frame%2==0 else self.vals_B
        in_h = self.heap_A if self.frame%2==0 else self.heap_B
        out_k = self.keys_B if self.frame%2==0 else self.keys_A
        out_v = self.vals_B if self.frame%2==0 else self.vals_A
        out_h = self.heap_B if self.frame%2==0 else self.heap_A
        
        cuda.memset_d32(out_k, 0xFFFFFFFF, self.hash_size * 2)
        
        self.kernel(in_k, in_v, in_h, out_k, out_v, out_h, 
                    np.int32(self.hash_size), np.float32(0.016),
                    block=(256,1,1), grid=(256,1))
        self.frame += 1

    def get_centroid(self):
        ptr = self.keys_A if self.frame%2==0 else self.keys_B
        cuda.memcpy_dtoh(self.host_keys, ptr)
        
        xs, ys, zs = [], [], []
        valid_keys = self.host_keys[self.host_keys != 0xFFFFFFFFFFFFFFFF]
        
        for k in valid_keys:
            val = int(k)
            x = (val & 0xFFFFF) - 10000
            y = ((val >> 20) & 0xFFFFF) - 10000
            z = ((val >> 40) & 0xFFFFF) - 10000
            xs.append(x)
            ys.append(y)
            zs.append(z)
            
        if len(xs) == 0: return [0,0,0]
        return [sum(xs)/len(xs), sum(ys)/len(ys), sum(zs)/len(zs)]

# --- 2. GENERATOR LOOP ---

def generate_dataset():
    sim = UmerEngine()
    sim.reset()
    sim.add_object(0, 0, 50, size=8) 
    
    print("🎥 STARTING SIMULATION & METADATA GENERATION...")
    
    dataset = []
    
    for i in range(100):
        sim.step()
        pos = sim.get_centroid()
        
        frame_data = {
            "frame_id": i,
            "object_transform": {
                "id": "battery_lithium_01",
                "pos": [pos[0]*0.1, pos[2]*0.1, pos[1]*0.1], 
                "rot": [random.uniform(0, 360), random.uniform(0, 360), random.uniform(0, 360)] 
            },
            "environment": {
                "sun_angle": random.uniform(20, 160), 
                "sun_color_temp": random.choice(["warm", "neutral", "cool"]),
                "weather": random.choices(["clear", "rain", "fog"], weights=[0.7, 0.2, 0.1])[0],
                "skybox_id": random.randint(1, 5)
            },
            "camera": {
                "distance": random.uniform(5.0, 15.0),
                "yaw": random.uniform(0, 360),
                "pitch": random.uniform(10, 45)
            }
        }
        
        dataset.append(frame_data)
        if i % 10 == 0: print(f"Processing Frame {i} | Centroid: {pos}")

    with open('umer_synthetic_data.json', 'w') as f:
        json.dump(dataset, f, indent=4)
        
    print("✅ DONE! Download 'umer_synthetic_data.json'")

if __name__ == "__main__":
    generate_dataset()

🎥 STARTING SIMULATION & METADATA GENERATION...
Processing Frame 0 | Centroid: [0.875, 0.84375, 52.625]
Processing Frame 10 | Centroid: [0.875, 0.84375, 52.625]
Processing Frame 20 | Centroid: [0.76, 0.96, 53.12]
Processing Frame 30 | Centroid: [0.64, 0.92, 52.12]
Processing Frame 40 | Centroid: [0.76, 0.92, 51.12]
Processing Frame 50 | Centroid: [0.6, 0.76, 50.12]
Processing Frame 60 | Centroid: [0.72, 0.88, 49.12]
Processing Frame 70 | Centroid: [0.64, 0.8, 47.12]
Processing Frame 80 | Centroid: [0.68, 0.76, 45.12]
Processing Frame 90 | Centroid: [0.56, 0.64, 43.12]
✅ DONE! Download 'umer_synthetic_data.json'


In [3]:
import json
import math

# 1. Load the "broken" JSON
input_filename = "umer_synthetic_data.json"
output_filename = "umer_synthetic_data_clean.json"

print(f"🔧 Repairing {input_filename}...")

try:
    with open(input_filename, 'r') as f:
        # We read it as raw text first to avoid crashing the python parser
        raw_data = f.read()
    
    # 2. Brute-force replace NaN with 0
    # Python writes NaN as NaN, but JSON expects null or a number.
    # We will just zero it out so Unity doesn't crash.
    clean_data = raw_data.replace('NaN', '0.0').replace('nan', '0.0')
    
    # 3. Verify it is valid JSON now
    parsed = json.loads(clean_data)
    
    # 4. Save the Clean File
    with open(output_filename, 'w') as f:
        json.dump(parsed, f, indent=4)
        
    print(f"✅ SUCCESS! Created '{output_filename}'.")
    print("👉 Drag THIS file into your Unity 'Data' folder.")

except Exception as e:
    print(f"❌ Error: {e}")

🔧 Repairing umer_synthetic_data.json...
✅ SUCCESS! Created 'umer_synthetic_data_clean.json'.
👉 Drag THIS file into your Unity 'Data' folder.


In [4]:
import json
import random

dataset = []
print("🎥 GENERATING RANDOMIZED DATASET (Action Mode)...")

for i in range(100):
    # Cat falls from 5m to 0m
    height = 5.0 - (i * 0.05) 
    if height < 0: height = 0

    frame_data = {
        "frame_id": i,
        "object_transform": {
            "id": "battery_lithium_01",
            "pos": [0, height, 0],       # Center of World
            "rot": [random.uniform(0, 30), random.uniform(0, 360), 0] # Random Spin
        },
        "environment": {
            "sun_angle": random.uniform(20, 160),
            "sun_color_temp": "neutral",
            "weather": "clear",
            "skybox_id": 1
        },
        "camera": {
            # RANDOM CAMERA IS BACK!
            "distance": random.uniform(3.0, 7.0), # Zoom in/out
            "yaw": random.uniform(0, 360),        # Spin around
            "pitch": random.uniform(10, 60)       # High and low angles
        }
    }
    dataset.append(frame_data)

# Save and Overwrite
with open('umer_synthetic_data.json', 'w') as f:
    json.dump(dataset, f, indent=4)

print("✅ DONE. Drag this new JSON into Unity and Press Play.")

🎥 GENERATING RANDOMIZED DATASET (Action Mode)...
✅ DONE. Drag this new JSON into Unity and Press Play.


In [5]:
import json
import random

dataset = []
print("🎥 GENERATING CLEAN FOOTBALL DATASET...")

for i in range(100):
    frame_data = {
        "frame_id": i,
        # THIS was the problem before. We force 'player_01' for ALL frames.
        "object_transform": {
            "id": "player_01", 
            "pos": [0, 0, 0],   
            "rot": [0, random.uniform(0, 360), 0]
        },
        "environment": {
            "sun_angle": random.uniform(20, 160),
            "sun_color_temp": "neutral",
            "weather": "clear",
            "skybox_id": 1
        },
        "camera": {
            "distance": random.uniform(18.0, 28.0), # Stadium view distance
            "yaw": random.uniform(0, 360),
            "pitch": random.uniform(30, 60)
        }
    }
    dataset.append(frame_data)

# Force overwrite the file with clean syntax
with open('umer_synthetic_data.json', 'w') as f:
    json.dump(dataset, f, indent=4)

print("✅ DONE. Your JSON file is fixed.")

🎥 GENERATING CLEAN FOOTBALL DATASET...
✅ DONE. Your JSON file is fixed.
